# 07 — Extract polygon spectra

Use this notebook after a flightline has completed reflectance products. It follows the same configure–run–inspect pattern as the active notebooks: call one orchestrator, print its structured result, then inventory the polygon tables.

## 1. Configure the flightline and polygons

Keep the flightline path canonical and point `polygon_path` at a vector file with a known CRS and stable identifiers. Confirm spatial overlap before a large extraction.

In [ ]:
from pathlib import Path
from pprint import pprint

from spectralbridge.paths import FlightlinePaths
from spectralbridge.polygons import run_polygon_pipeline_for_flightline

RUN = False
base_folder = Path("outputs/neon_notebook")
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
polygon_path = Path("data/aop_macrosystems_data_1_7_25.geojson")
paths = FlightlinePaths(base_folder, flight_stem)

In [ ]:
print(f"Flightline directory: {paths.flight_dir.resolve()}")
print(f"Flightline exists: {paths.flight_dir.exists()}")
print(f"Polygon file: {polygon_path.resolve()}")
print(f"Polygon file exists: {polygon_path.exists()}")

## 2. Run or resume extraction

`overwrite=False` preserves existing valid polygon products. The result reports the index, extracted tables, merged table, and any skip or failure information supplied by the orchestrator.

In [ ]:
result = None
if RUN:
    result = run_polygon_pipeline_for_flightline(
        paths,
        polygon_path,
        overwrite=False,
    )
    pprint(result)
else:
    print("Dry run. Confirm CRS and spatial overlap before setting RUN = True.")

## 3. Check extracted tables

Inspect filenames and sizes first, then open the merged polygon table with the DuckDB pattern in notebook 04.

In [ ]:
polygon_tables = sorted(paths.flight_dir.glob("*polygon*.parquet")) if paths.flight_dir.exists() else []
print(f"Polygon Parquet files: {len(polygon_tables)}")
for path in polygon_tables:
    print(f"  {path.name}: {path.stat().st_size:,} bytes")

## 4. What to verify

Confirm polygon identifiers, coordinate reference system, overlap counts, and expected sensor columns. A nonempty table is not sufficient evidence if the polygons were shifted, duplicated, or only marginally overlapping.